# Analyse des vulnérabilités ANSSI

Ce notebook charge le fichier CSV généré par le pipeline et produit plusieurs graphiques d'analyse.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df = pd.read_csv("donnees_enrichies.csv")

df["Score CVSS"] = pd.to_numeric(df["Score CVSS"], errors="coerce")
df["Score EPSS"] = pd.to_numeric(df["Score EPSS"], errors="coerce")

df_cvss = df.dropna(subset=["Score CVSS"])
df_epss = df.dropna(subset=["Score EPSS"])
df_deux = df.dropna(subset=["Score CVSS", "Score EPSS"])

print(f"Nombre de lignes chargées : {len(df)}")
print(f"Colonnes : {list(df.columns)}")

## Aperçu des données

In [ ]:
df.head(10)

In [ ]:
df.describe()

## Distribution des scores CVSS

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df_cvss["Score CVSS"], bins=10, color="steelblue", edgecolor="white")
plt.title("Distribution des scores CVSS")
plt.xlabel("Score CVSS")
plt.ylabel("Nombre de CVE")
plt.tight_layout()
plt.savefig("graphique_cvss.png")
plt.show()

## Distribution des scores EPSS

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df_epss["Score EPSS"], bins=10, color="coral", edgecolor="white")
plt.title("Distribution des scores EPSS")
plt.xlabel("Score EPSS")
plt.ylabel("Nombre de CVE")
plt.tight_layout()
plt.savefig("graphique_epss.png")
plt.show()

## Top 10 des éditeurs les plus touchés

In [ ]:
vendors = df["Éditeur (Vendor)"].dropna()
top_vendors = vendors.value_counts().head(10)

plt.figure(figsize=(10, 6))
top_vendors.plot(kind="bar", color="mediumseagreen", edgecolor="white")
plt.title("Top 10 des éditeurs les plus touchés")
plt.xlabel("Éditeur")
plt.ylabel("Nombre de lignes")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("graphique_vendors.png")
plt.show()

## Top 10 des produits les plus touchés

In [ ]:
produits = df["Produit"].dropna()
top_produits = produits.value_counts().head(10)

plt.figure(figsize=(10, 6))
top_produits.plot(kind="bar", color="mediumpurple", edgecolor="white")
plt.title("Top 10 des produits les plus touchés")
plt.xlabel("Produit")
plt.ylabel("Nombre de lignes")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("graphique_produits.png")
plt.show()

## Score CVSS vs Score EPSS

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_deux["Score CVSS"], df_deux["Score EPSS"], color="tomato", alpha=0.6)
plt.title("Score CVSS vs Score EPSS")
plt.xlabel("Score CVSS")
plt.ylabel("Score EPSS")
plt.tight_layout()
plt.savefig("graphique_cvss_vs_epss.png")
plt.show()

## Répartition des niveaux de sévérité

In [ ]:
severite = df["Base Severity"].dropna()
compte_severite = severite.value_counts()

plt.figure(figsize=(8, 8))
plt.pie(compte_severite, labels=compte_severite.index, autopct="%1.1f%%", startangle=140)
plt.title("Répartition des niveaux de sévérité")
plt.tight_layout()
plt.savefig("graphique_severite.png")
plt.show()

## Nombre de CVE par type de bulletin

In [ ]:
type_bulletin = df.groupby("Type")["Identifiant CVE"].count()

plt.figure(figsize=(8, 6))
type_bulletin.plot(kind="bar", color=["steelblue", "coral"], edgecolor="white")
plt.title("Nombre de CVE par type de bulletin")
plt.xlabel("Type de bulletin")
plt.ylabel("Nombre de CVE")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("graphique_type_bulletin.png")
plt.show()

## Machine Learning

### Modèle non supervisé — KMeans

On regroupe les CVEs en clusters selon leur score CVSS et EPSS, sans étiquettes prédéfinies.

In [ ]:
df_ml = df.dropna(subset=["Score CVSS", "Score EPSS"])
df_ml = df_ml[df_ml["Base Severity"] != "Non renseigné"].copy()

print(f"Lignes utilisées pour le ML : {len(df_ml)}")

#### Clustering KMeans

In [ ]:
X_kmeans = df_ml[["Score CVSS", "Score EPSS"]].copy()

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_ml["Cluster"] = kmeans.fit_predict(X_kmeans)

couleurs = ["steelblue", "coral", "mediumseagreen"]
plt.figure(figsize=(10, 6))
for i in range(3):
    groupe = df_ml[df_ml["Cluster"] == i]
    plt.scatter(groupe["Score CVSS"], groupe["Score EPSS"], label=f"Cluster {i}", color=couleurs[i], alpha=0.7)

plt.title("Clustering KMeans des CVEs")
plt.xlabel("Score CVSS")
plt.ylabel("Score EPSS")
plt.legend()
plt.tight_layout()
plt.savefig("graphique_kmeans.png")
plt.show()

In [ ]:
for i in range(3):
    groupe = df_ml[df_ml["Cluster"] == i]
    cvss_moy = groupe["Score CVSS"].mean()
    epss_moy = groupe["Score EPSS"].mean()
    print(f"Cluster {i} - {len(groupe)} CVE - CVSS moyen : {cvss_moy:.1f} - EPSS moyen : {epss_moy:.3f}")

### Modèle supervisé — Random Forest

On prédit la sévérité réelle  fournie par MITRE à partir du score EPSS et du type de faille CWE.

In [ ]:
le_cwe = LabelEncoder()
df_ml["CWE_encoded"] = le_cwe.fit_transform(df_ml["Type CWE"].fillna("Inconnu"))

le_severity = LabelEncoder()
df_ml["Severity_encoded"] = le_severity.fit_transform(df_ml["Base Severity"])

X = df_ml[["Score EPSS", "CWE_encoded"]]
y = df_ml["Severity_encoded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Rapport de classification :")
print(classification_report(y_test, y_pred, labels=list(range(len(le_severity.classes_))), target_names=le_severity.classes_, zero_division=0))

#### Importance des variables

In [ ]:
importances = clf.feature_importances_
noms_features = ["Score EPSS", "Type CWE"]

plt.figure(figsize=(8, 5))
plt.bar(noms_features, importances, color=["coral", "steelblue"])
plt.title("Importance des variables - Random Forest")
plt.ylabel("Importance")
plt.tight_layout()
plt.savefig("graphique_importance_features.png")
plt.show()